# Career Recommendation System

Trains and compares **Logistic Regression** vs **Random Forest**, picks the best model,
saves it, and exposes a `predict_api()` function ready to be called from a Flask/FastAPI backend.

In [ ]:
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
DATASET_NAME = "career_recommendation_dataset.csv"
MODEL_PATH = Path("career_recommendation_model.joblib")
TOP_K = 3

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def normalize_text(value) -> str:
    """Lowercase, strip punctuation separators, collapse whitespace."""
    if pd.isna(value):
        return ""
    text = str(value).lower()
    for ch in ";,/|":
        text = text.replace(ch, " ")
    return re.sub(r"\s+", " ", text).strip()


def load_data() -> pd.DataFrame:
    path = Path(DATASET_NAME)
    if not path.exists():
        path = Path.cwd() / DATASET_NAME
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find '{DATASET_NAME}'. Put the CSV in the same folder as this notebook."
        )

    df = pd.read_csv(path).copy()

    required_cols = {"Age", "Education", "Skills", "Interests", "Recommended_Career"}
    missing = required_cols.difference(df.columns)
    if missing:
        raise ValueError(f"Dataset is missing columns: {sorted(missing)}")

    df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
    df["Education"] = df["Education"].fillna("unknown").astype(str).str.strip()
    df["Skills"] = df["Skills"].fillna("").apply(normalize_text)
    df["Interests"] = df["Interests"].fillna("").apply(normalize_text)
    df["Recommended_Career"] = df["Recommended_Career"].fillna("").astype(str).str.strip()
    df["Text_Features"] = (df["Skills"] + " " + df["Interests"]).str.strip()
    return df

In [ ]:
# ── Preprocessor (shared by both models) ─────────────────────────────────────

def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "age",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                ["Age"],
            ),
            (
                "education",
                OneHotEncoder(handle_unknown="ignore"),
                ["Education"],
            ),
            (
                "text",
                TfidfVectorizer(max_features=4000, ngram_range=(1, 2), min_df=2),
                "Text_Features",
            ),
        ],
        remainder="drop",
    )


def build_lr_pipeline() -> Pipeline:
    return Pipeline([
        ("preprocessor", build_preprocessor()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            solver="lbfgs",
            C=1.0,
            random_state=RANDOM_STATE,
        )),
    ])


def build_rf_pipeline() -> Pipeline:
    return Pipeline([
        ("preprocessor", build_preprocessor()),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=4,
            min_samples_leaf=2,
            class_weight="balanced",   # handles any class imbalance
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────

df = load_data()
print(f"Shape: {df.shape}")
print(f"Classes ({df['Recommended_Career'].nunique()}): {sorted(df['Recommended_Career'].unique())}")
print("\nSamples per class:")
print(df["Recommended_Career"].value_counts().to_string())

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────

X = df[["Age", "Education", "Text_Features"]]
y = df["Recommended_Career"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [ ]:
# ── Train both models & compare ───────────────────────────────────────────────

candidates = {
    "Logistic Regression": build_lr_pipeline(),
    "Random Forest": build_rf_pipeline(),
}

results = {}

for name, pipeline in candidates.items():
    pipeline.fit(X_train, y_train)

    # Hold-out accuracy
    test_acc = accuracy_score(y_test, pipeline.predict(X_test))

    # 5-fold CV on training data (more reliable than a single split)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="accuracy", n_jobs=-1)

    results[name] = {
        "pipeline": pipeline,
        "test_acc": test_acc,
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
    }

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Hold-out accuracy : {test_acc:.4f}")
    print(f"  CV accuracy       : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print()
    print(classification_report(y_test, pipeline.predict(X_test), zero_division=0))

In [ ]:
# ── Pick best model (by CV mean, more robust than hold-out) ───────────────────

best_name = max(results, key=lambda n: results[n]["cv_mean"])
best_model = results[best_name]["pipeline"]

print(f"Best model: {best_name}")
print(f"  CV accuracy : {results[best_name]['cv_mean']:.4f} ± {results[best_name]['cv_std']:.4f}")
print(f"  Test accuracy: {results[best_name]['test_acc']:.4f}")

In [ ]:
# ── Save best model ───────────────────────────────────────────────────────────

joblib.dump(best_model, MODEL_PATH)
print(f"Saved: {MODEL_PATH}")

In [ ]:
# ── predict_api — call this from Flask / FastAPI ──────────────────────────────
#
# Input  : raw values exactly as your frontend sends them
# Output : list of dicts  [{"career": str, "score": float, "percent": str}, ...]
#
# Load once at server startup:
#   MODEL = joblib.load("career_recommendation_model.joblib")
# Then call:
#   predict_api(age, education, skills, interests, MODEL)

def predict_api(age, education, skills, interests, model=None, k=TOP_K):
    """Return top-k career recommendations as a list of dicts."""
    if model is None:
        model = joblib.load(MODEL_PATH)

    # Safely coerce age to float (frontend sends strings)
    try:
        age_val = float(age)
    except (ValueError, TypeError):
        age_val = np.nan

    sample = pd.DataFrame([{
        "Age": age_val,
        "Education": str(education).strip(),
        "Text_Features": f"{normalize_text(skills)} {normalize_text(interests)}".strip(),
    }])

    probs = model.predict_proba(sample)[0]
    top_idx = np.argsort(probs)[::-1][:k]

    return [
        {
            "career": model.classes_[i],
            "score": round(float(probs[i]), 4),
            "percent": f"{probs[i]*100:.1f}%",
        }
        for i in top_idx
    ]


# ── Quick smoke test ──────────────────────────────────────────────────────────

test_cases = [
    {"age": 21, "education": "Bachelor's", "skills": "design ui ux figma",       "interests": "backend web"},
    {"age": 24, "education": "Master's",   "skills": "python machine learning",   "interests": "ai deep learning"},
    {"age": 28, "education": "Bachelor's", "skills": "docker kubernetes aws ci",   "interests": "cloud infrastructure"},
]

for tc in test_cases:
    results_tc = predict_api(tc["age"], tc["education"], tc["skills"], tc["interests"], model=best_model)
    print(f"\nInput  → skills: '{tc['skills']}' | interests: '{tc['interests']}'")
    for r in results_tc:
        bar = "█" * int(r['score'] * 20)
        print(f"  {r['career']:<30} {r['percent']:>6}  {bar}")

## Flask integration snippet

Paste this into your `app.py`. The model is loaded once at startup — no re-loading on every request.

```python
from flask import Flask, request, jsonify
from flask_cors import CORS
import joblib
from notebook import predict_api   # or copy predict_api here directly

app = Flask(__name__)
CORS(app)  # allow frontend to call this

MODEL = joblib.load("career_recommendation_model.joblib")

@app.route("/recommend", methods=["POST"])
def recommend():
    data = request.get_json()
    results = predict_api(
        age=data.get("age"),
        education=data.get("education"),
        skills=data.get("skills"),
        interests=data.get("interests"),
        model=MODEL,
    )
    return jsonify({"recommendations": results})

if __name__ == "__main__":
    app.run(debug=True, port=5000)
```

Frontend fetch call:
```js
const res = await fetch("http://localhost:5000/recommend", {
  method: "POST",
  headers: { "Content-Type": "application/json" },
  body: JSON.stringify({ age, education, skills, interests })
});
const { recommendations } = await res.json();
// recommendations = [{career, score, percent}, ...]
```